# LeetCode 2244 — Minimum Rounds to Complete All Tasks bằng Python

Link đề: https://leetcode.com/problems/minimum-rounds-to-complete-all-tasks/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def minimumRounds(self, tasks: List[int]) -> int:
        ...
```

(Bài này giống hệt LC 2870 *Minimum Number of Operations to Make Array Empty*.)

## 1. Mục tiêu học

1. Nhận ra các mức độ khó **độc lập** với nhau ⇒ bài lớn tách thành nhiều bài con trên **tần suất** `v` của từng mức.
2. Giải bài con "biểu diễn `v` thành tổng các số 2 và 3 với **ít số hạng nhất**" theo ba cách: quy hoạch động kiểu đổi tiền (coin change), BFS, và **công thức đóng** `⌈v/3⌉`.
3. Chứng minh công thức `⌈v/3⌉` bằng lập luận **cận dưới + cách đạt được**, phân tích theo `v mod 3`.
4. Dùng `Counter`, sắp xếp + `groupby` để đếm tần suất; hiểu vì sao giá trị lớn (`10⁹`) không cản trở (ta không đánh chỉ số theo giá trị).
5. Viết test đối chiếu ngẫu nhiên với DP, xử lý trường hợp không thể (`v = 1`).

## 2. Đề bài & ví dụ

Cho mảng `tasks`, `tasks[i]` là **mức độ khó** của nhiệm vụ thứ `i`. Mỗi **vòng** (round) bạn hoàn thành **2 hoặc 3** nhiệm vụ **cùng mức độ khó**.

Trả về **số vòng ít nhất** để hoàn thành mọi nhiệm vụ, hoặc `-1` nếu không thể.

**Ví dụ 1:** `tasks = [2,2,3,3,2,4,4,4,4,4]` → `4`.
- Mức 2 có 3 nhiệm vụ: 1 vòng (3).
- Mức 3 có 2 nhiệm vụ: 1 vòng (2).
- Mức 4 có 5 nhiệm vụ: 2 vòng (3 + 2).

**Ví dụ 2:** `tasks = [2,3,3]` → `-1` (mức 2 chỉ có 1 nhiệm vụ — không vòng nào làm được 1).

**Ràng buộc:** `1 ≤ tasks.length ≤ 10⁵`, `1 ≤ tasks[i] ≤ 10⁹`.

In [1]:
from typing import List
from collections import Counter, deque
from itertools import groupby
from functools import lru_cache
import random, time, math

examples = [([2, 2, 3, 3, 2, 4, 4, 4, 4, 4], 4), ([2, 3, 3], -1)]
for tasks, exp in examples:
    print(tasks, "-> tần suất", dict(Counter(tasks)), "-> kỳ vọng", exp)

[2, 2, 3, 3, 2, 4, 4, 4, 4, 4] -> tần suất {2: 3, 3: 2, 4: 5} -> kỳ vọng 4
[2, 3, 3] -> tần suất {2: 1, 3: 2} -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 `Counter` — đếm tần suất

`Counter(tasks)` cho `dict` *mức độ khó → số nhiệm vụ*. Ta chỉ cần các **giá trị** (`.values()`), không cần biết mức độ khó cụ thể là bao nhiêu.

In [2]:
cnt = Counter([2, 2, 3, 3, 2, 4, 4, 4, 4, 4])
print(cnt, list(cnt.values()))

Counter({4: 5, 2: 3, 3: 2}) [3, 2, 5]


### 3.2 Chia lấy nguyên, chia lấy dư, và làm tròn lên

`v // 3` (thương), `v % 3` (dư), `⌈v/3⌉ = (v + 2) // 3 = -(-v // 3)`. Tránh `math.ceil(v / 3)` với số rất lớn vì phép chia `float` có thể làm tròn sai khi `v > 2⁵³` (ở bài này `v ≤ 10⁵` nên an toàn, nhưng nên quen cách nguyên).

In [3]:
for v in [2, 3, 4, 5, 6, 7, 8]:
    print(f"v={v}: v//3={v // 3}, v%3={v % 3}, ceil={(v + 2) // 3}, -(-v//3)={-(-v // 3)}")
big = 10**17 + 1
print("ceil float:", math.ceil(big / 3), "| ceil nguyên:", (big + 2) // 3)

v=2: v//3=0, v%3=2, ceil=1, -(-v//3)=1
v=3: v//3=1, v%3=0, ceil=1, -(-v//3)=1
v=4: v//3=1, v%3=1, ceil=2, -(-v//3)=2
v=5: v//3=1, v%3=2, ceil=2, -(-v//3)=2
v=6: v//3=2, v%3=0, ceil=2, -(-v//3)=2
v=7: v//3=2, v%3=1, ceil=3, -(-v//3)=3
v=8: v//3=2, v%3=2, ceil=3, -(-v//3)=3
ceil float: 33333333333333332 | ceil nguyên: 33333333333333334


Dòng cuối cho thấy `math.ceil(big/3)` (float) **khác** kết quả nguyên đúng — một lý do nữa để dùng số học nguyên.

### 3.3 Quy hoạch động kiểu "đổi tiền" (coin change)

`f(v)` = số đồng ít nhất (mệnh giá 2 và 3) để tạo tổng `v`; `f(0) = 0`; `f(v) = 1 + min(f(v − 2), f(v − 3))`; không tạo được ⇒ `∞`.

In [4]:
INF = float("inf")
def coin_table(N):
    f = [INF] * (N + 1); f[0] = 0
    for v in range(1, N + 1):
        best = INF
        if v >= 2: best = min(best, f[v - 2] + 1)
        if v >= 3: best = min(best, f[v - 3] + 1)
        f[v] = best
    return f
print(coin_table(12))

[0, inf, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4]


### 3.4 Sắp xếp + `groupby` để đếm (không dùng băm)

Sau khi sắp xếp, các giá trị bằng nhau liền nhau; `groupby` cho từng khối.

In [5]:
print([(k, len(list(g))) for k, g in groupby(sorted([2, 2, 3, 3, 2, 4, 4, 4, 4, 4]))])

[(2, 3), (3, 2), (4, 5)]


## 4. Ý tưởng nền tảng / trực giác

**Tách bài:** một vòng chỉ làm nhiệm vụ **cùng** mức độ khó ⇒ các mức độ khó không ảnh hưởng nhau. Đáp án = tổng số vòng ít nhất của từng mức; nếu **một** mức không làm được thì cả bài `-1`.

**Bài con:** có `v` nhiệm vụ cùng mức; cần viết `v = 2a + 3b` (`a, b ≥ 0`) với `a + b` nhỏ nhất.

Tính tay vài giá trị (nên dùng nhiều số 3 vì "mỗi vòng làm được nhiều hơn"):

| v | cách tốt nhất | số vòng |
|---|---|---|
| 1 | không có | -1 |
| 2 | 2 | 1 |
| 3 | 3 | 1 |
| 4 | 2+2 | 2 |
| 5 | 3+2 | 2 |
| 6 | 3+3 | 2 |
| 7 | 3+2+2 | 3 |
| 8 | 3+3+2 | 3 |
| 9 | 3+3+3 | 3 |
| 10 | 3+3+2+2 | 4 |

Nhìn cột cuối: `1, 1, 2, 2, 2, 3, 3, 3, 4` cho `v = 2..10` — chính là `⌈v/3⌉`! Quy luật theo `v mod 3`:
- dư 0: toàn số 3 ⇒ `v/3`.
- dư 2: thêm một số 2 ⇒ `v//3 + 1`.
- dư 1: "đổi" một số 3 thành hai số 2 (`3 + 1 = 2 + 2`) ⇒ `(v//3 − 1) + 2 = v//3 + 1`.

In [6]:
f = coin_table(20)
for v in range(1, 21):
    print(f"v={v:>2}: DP={f[v]!s:>3}  ceil(v/3)={(v + 2) // 3}  {'(không thể)' if f[v] == INF else ''}")

v= 1: DP=inf  ceil(v/3)=1  (không thể)
v= 2: DP=  1  ceil(v/3)=1  
v= 3: DP=  1  ceil(v/3)=1  
v= 4: DP=  2  ceil(v/3)=2  
v= 5: DP=  2  ceil(v/3)=2  
v= 6: DP=  2  ceil(v/3)=2  
v= 7: DP=  3  ceil(v/3)=3  
v= 8: DP=  3  ceil(v/3)=3  
v= 9: DP=  3  ceil(v/3)=3  
v=10: DP=  4  ceil(v/3)=4  
v=11: DP=  4  ceil(v/3)=4  
v=12: DP=  4  ceil(v/3)=4  
v=13: DP=  5  ceil(v/3)=5  
v=14: DP=  5  ceil(v/3)=5  
v=15: DP=  5  ceil(v/3)=5  
v=16: DP=  6  ceil(v/3)=6  
v=17: DP=  6  ceil(v/3)=6  
v=18: DP=  6  ceil(v/3)=6  
v=19: DP=  7  ceil(v/3)=7  
v=20: DP=  7  ceil(v/3)=7  


## 5. Cách 1 — Brute Force / baseline: thử mọi số vòng-3 cho từng mức

**Ý tưởng:** với tần suất `v`, thử mọi `b = 0..v//3` (số vòng làm 3); phần còn lại `v − 3b` phải chia hết cho 2 ⇒ `a = (v − 3b)/2`. Lấy `a + b` nhỏ nhất.

In [7]:
def brute_rounds(v):
    best = -1
    for b in range(v // 3 + 1):
        rest = v - 3 * b
        if rest % 2 == 0:
            r = b + rest // 2
            if best == -1 or r < best:
                best = r
    return best

def brute(tasks):
    total = 0
    for v in Counter(tasks).values():
        r = brute_rounds(v)
        if r == -1:
            return -1
        total += r
    return total

for tasks, exp in examples:
    got = brute(tasks)
    print(got, "OK" if got == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp:** đếm O(n); với mỗi tần suất `v` tốn O(v); `Σ v = n` ⇒ tổng **O(n)**! Thật ra "brute force" này đã đủ nhanh vì tổng tần suất bằng `n`. Nhưng nó vẫn có **nút thắt về tư duy**: ta lặp mà không hiểu cấu trúc; nếu đề đổi thành "`v` cho trước tới 10¹⁸" (một mức với số lượng khổng lồ, ví dụ cho dạng `(giá trị, số lần)`) thì O(v) không chạy được.

### 5.1 Baseline "rất thô": vét cạn đệ quy không nhớ trên `v`

`g(v) = 1 + min(g(v−2), g(v−3))` không ghi nhớ ⇒ số lời gọi tăng **theo hàm mũ**.

In [8]:
def naive_rec(v, counter):
    counter[0] += 1
    if v == 0: return 0
    if v == 1: return INF
    best = naive_rec(v - 2, counter) + 1
    if v >= 3: best = min(best, naive_rec(v - 3, counter) + 1)
    return best

for v in [10, 20, 30, 40]:
    c = [0]; r = naive_rec(v, c)
    print(f"v={v}: kết quả {r}, số lời gọi {c[0]}")

v=10: kết quả 4, số lời gọi 27
v=20: kết quả 7, số lời gọi 464
v=30: kết quả 10, số lời gọi 7738
v=40: kết quả 14, số lời gọi 128800


Số lời gọi tăng xấp xỉ theo dãy kiểu Padovan (~1,32ᵛ) ⇒ ghi nhớ hoặc công thức là bắt buộc nếu `v` lớn.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý:** với `v ≥ 2`, số vòng ít nhất là `⌈v/3⌉`; với `v = 1` là không thể.

**Cận dưới:** mỗi vòng làm tối đa 3 nhiệm vụ ⇒ cần ít nhất `⌈v/3⌉` vòng.

**Đạt được** (với `v ≥ 2`), xét `r = v mod 3`, `q = v // 3`:
- `r = 0`: `q` vòng-3 ⇒ `q = ⌈v/3⌉` vòng.
- `r = 2`: `q` vòng-3 + 1 vòng-2 ⇒ `q + 1 = ⌈v/3⌉`.
- `r = 1`: vì `v ≥ 2` và `v ≡ 1` ⇒ `v ≥ 4` ⇒ `q ≥ 1`. Dùng `q − 1` vòng-3 + 2 vòng-2: `3(q−1) + 4 = 3q + 1 = v` ✔ ⇒ `q + 1 = ⌈v/3⌉` vòng.

**Quy tắc "nếu… thì…":**
- **Nếu** có tần suất bằng 1 **thì** trả `-1` ngay.
- **Ngược lại** cộng `(v + 2) // 3` cho mỗi tần suất.

**Tham lam ngây thơ "làm 3 khi còn ≥ 3"** có đúng không? Với `v = 4`: làm 3 còn 1 ⇒ kẹt! Tham lam phải có "lùi một bước" (đổi 3+1 thành 2+2). Công thức `⌈v/3⌉` đã gói sẵn bước đổi này — xem Lỗi 2 ở mục 13.

In [9]:
def plan(v):
    # trả về một cách chia cụ thể (danh sách số 2/3)
    if v == 1: return None
    q, r = divmod(v, 3)
    if r == 0: return [3] * q
    if r == 2: return [3] * q + [2]
    return [3] * (q - 1) + [2, 2]

for v in range(2, 12):
    p = plan(v)
    print(v, p, "tổng", sum(p), "số vòng", len(p), "= ceil:", len(p) == (v + 2) // 3)

2 [2] tổng 2 số vòng 1 = ceil: True
3 [3] tổng 3 số vòng 1 = ceil: True
4 [2, 2] tổng 4 số vòng 2 = ceil: True
5 [3, 2] tổng 5 số vòng 2 = ceil: True
6 [3, 3] tổng 6 số vòng 2 = ceil: True
7 [3, 2, 2] tổng 7 số vòng 3 = ceil: True
8 [3, 3, 2] tổng 8 số vòng 3 = ceil: True
9 [3, 3, 3] tổng 9 số vòng 3 = ceil: True
10 [3, 3, 2, 2] tổng 10 số vòng 4 = ceil: True
11 [3, 3, 3, 2] tổng 11 số vòng 4 = ceil: True


### 6.1 Góc nhìn "bài toán đồng xu Frobenius"

Với hai mệnh giá nguyên tố cùng nhau `p, q`, số lớn nhất **không** biểu diễn được dưới dạng `ap + bq` (`a, b ≥ 0`) là `pq − p − q` (định lý Sylvester). Với `p = 2, q = 3`: `6 − 2 − 3 = 1` ⇒ **mọi** `v ≥ 2` đều biểu diễn được, chỉ `v = 1` (và `v = 0` tầm thường) là ngoại lệ. Đó là lý do bài chỉ có đúng một trường hợp `-1`.

Nếu đề đổi mệnh giá thành `{3, 5}` thì các số không biểu diễn được là `1, 2, 4, 7` (số lớn nhất `15 − 3 − 5 = 7`) — khi đó điều kiện `-1` phức tạp hơn và nên dùng DP (biến thể 15.2). Kiểm chứng nhanh:

In [10]:
def unrepresentable(p, q, limit=60):
    ok = [False] * (limit + 1); ok[0] = True
    for x in range(1, limit + 1):
        ok[x] = (x >= p and ok[x - p]) or (x >= q and ok[x - q])
    return [x for x in range(1, limit + 1) if not ok[x]]

for p, q in [(2, 3), (3, 5), (4, 7)]:
    bad = unrepresentable(p, q)
    print(f"mệnh giá {{{p}, {q}}}: không biểu diễn được {bad}, lớn nhất = {max(bad)}, công thức pq-p-q = {p*q - p - q}")

mệnh giá {2, 3}: không biểu diễn được [1], lớn nhất = 1, công thức pq-p-q = 1
mệnh giá {3, 5}: không biểu diễn được [1, 2, 4, 7], lớn nhất = 7, công thức pq-p-q = 7
mệnh giá {4, 7}: không biểu diễn được [1, 2, 3, 5, 6, 9, 10, 13, 17], lớn nhất = 17, công thức pq-p-q = 17


### 6.2 Vì sao "nhiều số 3 nhất" là đúng hướng?

Thay hai vòng-2 (4 nhiệm vụ, 2 vòng) … không thể bằng vòng-3 vì 4 không chia hết cho 3; nhưng thay **ba vòng-2** (6 nhiệm vụ, 3 vòng) bằng **hai vòng-3** (6 nhiệm vụ, 2 vòng) luôn tiết kiệm 1 vòng. Vậy một lời giải tối ưu có **tối đa hai vòng-2** (nếu có ≥ 3 vòng-2 thì đổi được). Với ≤ 2 vòng-2, phần còn lại là bội của 3 ⇒ số vòng-2 được quyết định bởi `v mod 3`: dư 0 → 0 vòng-2, dư 2 → 1 vòng-2, dư 1 → 2 vòng-2 (vì `2·2 = 4 ≡ 1`). Đây là một chứng minh thứ hai cho công thức, theo kiểu **đổi chỗ (exchange argument)**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP đổi tiền dùng chung một bảng cho mọi tần suất

Dựng bảng `f` tới `max(tần suất)` một lần (O(max v)), rồi tra cho từng tần suất.

In [11]:
def dp_table(tasks):
    vals = list(Counter(tasks).values())
    f = coin_table(max(vals))
    total = 0
    for v in vals:
        if f[v] == INF:
            return -1
        total += f[v]
    return total

for tasks, exp in examples:
    got = dp_table(tasks)
    print(got, "OK" if got == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp cách 2:** O(n + max v) = O(n) thời gian, O(max v) bộ nhớ.

### 7.2 Cách 3 — BFS trên "số nhiệm vụ còn lại"

Từ `v`, mỗi bước trừ 2 hoặc 3; BFS tới 0 cho số bước ít nhất. Minh họa rằng bài con là một **đường đi ngắn nhất** trên đồ thị số nguyên.

In [12]:
def bfs_rounds(v):
    dist = {v: 0}; q = deque([v])
    while q:
        x = q.popleft()
        if x == 0: return dist[x]
        for step in (3, 2):
            y = x - step
            if y >= 0 and y not in dist:
                dist[y] = dist[x] + 1; q.append(y)
    return -1

def bfs_all(tasks):
    total = 0
    for v in Counter(tasks).values():
        r = bfs_rounds(v)
        if r == -1: return -1
        total += r
    return total

for tasks, exp in examples:
    print(bfs_all(tasks), "OK" if bfs_all(tasks) == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp cách 3:** O(v) mỗi tần suất ⇒ O(n) tổng, nhưng hằng số lớn (dict, deque).

### 7.3 Cách 4 — Công thức `⌈v/3⌉` (cách chuẩn)

In [13]:
def formula(tasks):
    ans = 0
    for v in Counter(tasks).values():
        if v == 1:
            return -1
        ans += (v + 2) // 3
    return ans

for tasks, exp in examples:
    got = formula(tasks)
    print(got, "OK" if got == exp else "SAI")

4 OK
-1 OK


Bản doocs viết `v // 3 + (v % 3 != 0)` — tương đương `⌈v/3⌉` vì `True` là 1.

**Độ phức tạp cách 4:** O(n) để đếm, O(số mức khác nhau) để cộng; bộ nhớ O(số mức khác nhau).

### 7.4 Cách 5 — Sắp xếp + `groupby` (không dùng bảng băm)

Hữu ích khi muốn O(1) bộ nhớ phụ (sắp xếp tại chỗ) hoặc khi băm bị cấm.

In [14]:
def sort_group(tasks):
    ans = 0
    for _, g in groupby(sorted(tasks)):
        v = sum(1 for _ in g)
        if v == 1: return -1
        ans += (v + 2) // 3
    return ans

for tasks, exp in examples:
    print(sort_group(tasks), "OK" if sort_group(tasks) == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp cách 5:** O(n log n) thời gian, O(n) cho bản sắp xếp (hoặc O(1) thêm nếu `tasks.sort()` tại chỗ).

### 7.5 Cách 6 — Một lượt với hai con trỏ trên mảng đã sắp xếp

Không cần `groupby`: đếm độ dài mỗi khối bằng con trỏ `j` chạy tới hết khối.

In [15]:
def two_pointers_sorted(tasks):
    a = sorted(tasks); n = len(a); i = 0; ans = 0
    while i < n:
        j = i
        while j < n and a[j] == a[i]:
            j += 1
        v = j - i
        if v == 1: return -1
        ans += (v + 2) // 3
        i = j
    return ans

for tasks, exp in examples:
    print(two_pointers_sorted(tasks), "OK" if two_pointers_sorted(tasks) == exp else "SAI")

4 OK
-1 OK


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1, kèm cách chia cụ thể cho mỗi mức:

| mức | v | v mod 3 | cách chia | số vòng | tổng tích lũy |
|---|---|---|---|---|---|
| 2 | 3 | 0 | 3 | 1 | 1 |
| 3 | 2 | 2 | 2 | 1 | 2 |
| 4 | 5 | 2 | 3 + 2 | 2 | 4 |

In [16]:
def trace(tasks):
    cnt = Counter(tasks); ans = 0
    for level, v in cnt.items():
        if v == 1:
            print(f"mức {level}: v=1 -> không thể, trả -1"); return -1
        r = (v + 2) // 3; ans += r
        print(f"mức {level}: v={v}, v%3={v % 3}, chia {plan(v)} -> {r} vòng, tổng = {ans}")
    return ans

print("Kết quả:", trace([2, 2, 3, 3, 2, 4, 4, 4, 4, 4]))
print()
print("Kết quả:", trace([5, 5, 5, 5, 7, 7, 7, 7, 7, 7, 7, 9, 9]))

mức 2: v=3, v%3=0, chia [3] -> 1 vòng, tổng = 1
mức 3: v=2, v%3=2, chia [2] -> 1 vòng, tổng = 2
mức 4: v=5, v%3=2, chia [3, 2] -> 2 vòng, tổng = 4
Kết quả: 4

mức 5: v=4, v%3=1, chia [2, 2] -> 2 vòng, tổng = 2
mức 7: v=7, v%3=1, chia [3, 2, 2] -> 3 vòng, tổng = 5
mức 9: v=2, v%3=2, chia [2] -> 1 vòng, tổng = 6
Kết quả: 6


Trace DP (cách 2) cho tần suất 7, in quyết định ở mỗi ô:

In [17]:
def trace_dp(v):
    f = [INF] * (v + 1); f[0] = 0; choice = [None] * (v + 1)
    for x in range(1, v + 1):
        c2 = f[x - 2] + 1 if x >= 2 else INF
        c3 = f[x - 3] + 1 if x >= 3 else INF
        f[x] = min(c2, c3); choice[x] = 3 if c3 <= c2 and c3 < INF else (2 if c2 < INF else None)
        print(f"f[{x}] = min(f[{x}-2]+1={c2}, f[{x}-3]+1={c3}) = {f[x]}  (chọn {choice[x]})")
    parts = []; x = v
    while x > 0: parts.append(choice[x]); x -= choice[x]
    print("truy vết:", parts)

trace_dp(7)

f[1] = min(f[1-2]+1=inf, f[1-3]+1=inf) = inf  (chọn None)
f[2] = min(f[2-2]+1=1, f[2-3]+1=inf) = 1  (chọn 2)
f[3] = min(f[3-2]+1=inf, f[3-3]+1=1) = 1  (chọn 3)
f[4] = min(f[4-2]+1=2, f[4-3]+1=inf) = 2  (chọn 2)
f[5] = min(f[5-2]+1=2, f[5-3]+1=2) = 2  (chọn 3)
f[6] = min(f[6-2]+1=3, f[6-3]+1=2) = 2  (chọn 3)
f[7] = min(f[7-2]+1=3, f[7-3]+1=3) = 3  (chọn 3)
truy vết: [3, 2, 2]


## 9. Code nộp LeetCode

In [18]:
from typing import List
from collections import Counter

class Solution:
    def minimumRounds(self, tasks: List[int]) -> int:
        ans = 0
        for v in Counter(tasks).values():
            if v == 1:
                return -1              # không vòng nào làm được đúng 1 nhiệm vụ
            ans += (v + 2) // 3        # ceil(v / 3)
        return ans

In [19]:
sol = Solution()
for tasks, exp in examples:
    got = sol.minimumRounds(tasks)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

4 | kỳ vọng 4 | OK
-1 | kỳ vọng -1 | OK


## 10. Vì sao thuật toán đúng

**Bước 1 — tách bài.** Mỗi vòng chỉ chạm một mức độ khó. Vì vậy một lịch làm việc cho cả mảng tương ứng một-một với một bộ lịch riêng cho từng mức. Tổng số vòng là tổng các thành phần ⇒ tối thiểu tổng = tổng các tối thiểu, và nếu có mức không có lịch nào thì không có lịch tổng.

**Bước 2 — bài con.** `v = 1`: mỗi vòng làm ≥ 2 nhiệm vụ nên không thể. `v ≥ 2`: cận dưới `⌈v/3⌉` (mỗi vòng ≤ 3) và cách đạt được theo `v mod 3` (mục 6).

**Bước 3 — kiểm tra trên mọi `v`.** Cell dưới so công thức với DP cho mọi `v ≤ 5000` — một bằng chứng thực nghiệm bổ sung cho chứng minh.

In [20]:
f = coin_table(5000)
assert all((f[v] == INF) == (v == 1) for v in range(1, 5001))
assert all(f[v] == (v + 2) // 3 for v in range(2, 5001))
print("công thức ceil(v/3) khớp DP với mọi v trong [2, 5000]")

công thức ceil(v/3) khớp DP với mọi v trong [2, 5000]


## 11. Phân tích Time / Space Complexity

`n = len(tasks)`, `D` = số mức độ khó khác nhau, `V = max tần suất ≤ n`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử số vòng-3 | O(n) (vì Σv = n) | O(D) |
| 1b. Đệ quy không nhớ | mũ theo v | O(v) |
| 2. DP bảng chung | O(n + V) | O(V + D) |
| 3. BFS | O(n) hằng số lớn | O(V) |
| 4. Công thức | O(n) | O(D) |
| 5/6. Sắp xếp | O(n log n) | O(n) |

In [21]:
rng = random.Random(2244)
big = [rng.randint(1, 3000) for _ in range(10**5)]
for name, f in [("brute", brute), ("dp_table", dp_table), ("bfs_all", bfs_all), ("formula", formula),
                ("sort_group", sort_group), ("two_pointers_sorted", two_pointers_sorted), ("Solution", Solution().minimumRounds)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<20} -> {r}  {dt*1000:.0f} ms")

brute                -> 34341  5 ms
dp_table             -> 34341  3 ms
bfs_all              -> 34341  17 ms
formula              -> 34341  3 ms
sort_group           -> 34341  17 ms


two_pointers_sorted  -> 34341  21 ms
Solution             -> 34341  3 ms


(Với 3000 mức ngẫu nhiên trên 10⁵ phần tử, rất có thể có mức chỉ xuất hiện 1 lần ⇒ `-1`; cell sau thử bộ dữ liệu chắc chắn khả thi.)

In [22]:
ok_big = [x for x in range(1, 20001) for _ in range(5)]     # mỗi mức 5 lần -> 2 vòng mỗi mức
rng.shuffle(ok_big)
print(Solution().minimumRounds(ok_big), "= 20000 * 2 =", 40000)

40000

 = 20000 * 2 = 40000


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [23]:
ALL = {"brute": brute, "dp_table": dp_table, "bfs_all": bfs_all, "formula": formula,
       "sort_group": sort_group, "two_pointers_sorted": two_pointers_sorted, "Solution": Solution().minimumRounds}
edge = examples + [
    ([1], -1),
    ([1, 1], 1),
    ([1, 1, 1], 1),
    ([1, 1, 1, 1], 2),                         # 2 + 2 (không phải 3 + 1)
    ([7] * 7, 3),                              # 3 + 2 + 2
    ([10**9] * 6 + [1] * 2, 3),                # giá trị lớn không sao
    ([5, 5, 5, 5, 6], -1),                     # 6 chỉ 1 lần
    ([1, 2, 1, 2, 1, 2, 1, 2], 4),             # mỗi mức 4 lần -> 2 + 2
]
for tasks, exp in edge:
    for name, f in ALL.items():
        a = list(tasks)
        assert f(a) == exp, (name, tasks, f(list(tasks)), exp)
        assert a == tasks
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Sinh mảng có nhiều mức lặp lại (bảng giá trị nhỏ) và cả mảng có mức xuất hiện đúng 1 lần.

In [24]:
rng = random.Random(42)
cnt_neg = 0
for _ in range(3000):
    n = rng.randint(1, 40); hi = rng.choice([1, 2, 3, 5, 10, 10**9])
    tasks = [rng.randint(1, hi) for _ in range(n)]
    ref = brute(tasks); cnt_neg += ref == -1
    for name, f in ALL.items():
        assert f(list(tasks)) == ref, (name, tasks)
print("3000 case OK; số case trả -1:", cnt_neg)

3000 case OK; số case trả -1: 1265


### 12.3 Kiểm tra bằng cách dựng lịch thật

Với mỗi mức dựng lịch theo `plan(v)`; kiểm tra: mỗi vòng 2 hoặc 3 nhiệm vụ cùng mức, mọi nhiệm vụ được làm đúng một lần, số vòng = đáp án.

In [25]:
def build_schedule(tasks):
    sched = []
    for level, v in Counter(tasks).items():
        p = plan(v)
        if p is None: return None
        sched += [(level, k) for k in p]
    return sched

rng = random.Random(7)
for _ in range(500):
    tasks = [rng.randint(1, 6) for _ in range(rng.randint(1, 60))]
    s = build_schedule(tasks); ans = Solution().minimumRounds(tasks)
    if s is None:
        assert ans == -1
    else:
        assert len(s) == ans and all(k in (2, 3) for _, k in s)
        done = Counter()
        for level, k in s: done[level] += k
        assert done == Counter(tasks)
print("dựng lịch thật: OK")

dựng lịch thật: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `v // 3` (làm tròn xuống) hoặc `v // 2`

In [26]:
def wrong_floor(tasks):
    ans = 0
    for v in Counter(tasks).values():
        if v == 1: return -1
        ans += v // 3                          # SAI: v = 2 ra 0 vòng
    return ans

print("bản sai :", wrong_floor([2, 2, 3, 3, 2, 4, 4, 4, 4, 4]))
print("bản đúng:", Solution().minimumRounds([2, 2, 3, 3, 2, 4, 4, 4, 4, 4]))

bản sai : 2
bản đúng: 4


**Sửa:** cần **làm tròn lên**: `(v + 2) // 3`.

### Lỗi 2 — Tham lam "làm 3 khi còn ≥ 3, rồi làm 2" mà không lùi bước

In [27]:
def wrong_greedy(tasks):
    ans = 0
    for v in Counter(tasks).values():
        while v >= 3:
            v -= 3; ans += 1
        if v == 2: ans += 1
        elif v == 1: return -1                 # SAI: v = 4 -> 3 + 1 -> kẹt
    return ans

print("bản sai :", wrong_greedy([1, 1, 1, 1]))
print("bản đúng:", Solution().minimumRounds([1, 1, 1, 1]))

bản sai : -1
bản đúng: 2


**Vì sao sai:** với `v = 4, 7, 10, …` (dư 1), lấy 3 tới cùng sẽ còn lại 1. Phải "trả lại" một vòng-3 để thành hai vòng-2. **Sửa:** công thức `⌈v/3⌉`, hoặc tham lam có điều kiện "làm 3 khi còn ≥ 3 **và** phần còn lại khác 1".

### Lỗi 3 — Trả `-1` khi **tổng** `n = 1` thay vì khi **một mức** có tần suất 1

In [28]:
def wrong_check_total(tasks):
    if len(tasks) == 1: return -1
    return sum((v + 2) // 3 for v in Counter(tasks).values())     # SAI: bỏ qua mức lẻ loi

print("bản sai :", wrong_check_total([2, 3, 3]))
print("bản đúng:", Solution().minimumRounds([2, 3, 3]))

bản sai : 2
bản đúng: -1


**Sửa:** kiểm tra `v == 1` cho **từng** tần suất.

### Lỗi 4 — Dùng mảng đếm theo giá trị `cnt = [0] * (max(tasks) + 1)`

In [29]:
def wrong_count_array(tasks):
    try:
        if max(tasks) > 10**7:
            raise MemoryError(f"cần mảng {max(tasks) + 1} phần tử")
        cnt = [0] * (max(tasks) + 1)
        for x in tasks: cnt[x] += 1
        return sum((v + 2) // 3 for v in cnt if v) if all(v != 1 for v in cnt) else -1
    except MemoryError as e:
        return f"MemoryError (mô phỏng): {e}"

print("bản sai :", wrong_count_array([10**9, 10**9]))
print("bản đúng:", Solution().minimumRounds([10**9, 10**9]))

bản sai : MemoryError (mô phỏng): cần mảng 1000000001 phần tử
bản đúng: 1


**Vì sao sai:** `tasks[i]` tới `10⁹` ⇒ mảng 10⁹ ô (vài GB) — tràn bộ nhớ (ta mô phỏng bằng cách kiểm tra trước, không cấp phát thật). **Sửa:** `Counter`/`dict` (chỉ lưu giá trị xuất hiện) hoặc sắp xếp.

### Lỗi 5 — Dùng `math.ceil(v / 3)` với số rất lớn (biến thể đếm khổng lồ)

In [30]:
def ceil_float(v): return math.ceil(v / 3)
def ceil_int(v): return (v + 2) // 3
v = 3 * 10**17 + 1
print("float:", ceil_float(v), "| nguyên:", ceil_int(v), "| khác nhau:", ceil_float(v) != ceil_int(v))

float: 100000000000000000 | nguyên: 100000000000000001 | khác nhau: True


Ở bài gốc `v ≤ 10⁵` nên `math.ceil(v/3)` vẫn đúng, nhưng thói quen dùng số học nguyên tránh lỗi khi đề đổi (biến thể 15.1). **Sửa:** `(v + 2) // 3` hoặc `-(-v // 3)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi thao tác chỉ chạm một loại" ⇒ **đếm tần suất** rồi giải từng loại độc lập.
- Biểu diễn `v` bằng các số `{2, 3}` ít số hạng nhất ⇒ `⌈v/3⌉` (trừ `v = 1`). Tổng quát hơn với mệnh giá tùy ý ⇒ DP đổi tiền.
- Khi nghi ngờ công thức: viết DP cho `v ≤ vài nghìn` và so sánh — 3 dòng code.
- Giá trị lớn (`10⁹`) ⇒ băm, không dùng mảng đếm theo giá trị.
- Bài cùng họ: 2870 (giống hệt), 322 Coin Change, 279 Perfect Squares, 2384, 1647.

## 15. Biến thể của bài

### 15.1 Biến thể A — Input dạng nén `(mức độ khó, số lượng)` với số lượng tới 10¹⁸

Vòng lặp O(v) không còn dùng được, nhưng công thức nguyên vẫn O(1) mỗi mức.

In [31]:
def rounds_compressed(pairs):
    total = Counter()
    for level, c in pairs: total[level] += c
    ans = 0
    for v in total.values():
        if v == 1: return -1
        ans += (v + 2) // 3
    return ans

print(rounds_compressed([(5, 10**18), (7, 4), (5, 2)]), rounds_compressed([(1, 1)]))

333333333333333336 -1


### 15.2 Biến thể B — Mỗi vòng làm được `a` **hoặc** `b` nhiệm vụ (mệnh giá tổng quát)

Công thức đóng không còn đơn giản; dùng DP đổi tiền cho mỗi tần suất (bảng chung tới `max v`).

In [32]:
def rounds_general(tasks, sizes):
    vals = list(Counter(tasks).values()); V = max(vals)
    f = [INF] * (V + 1); f[0] = 0
    for x in range(1, V + 1):
        f[x] = min((f[x - s] + 1 for s in sizes if s <= x), default=INF)
    total = 0
    for v in vals:
        if f[v] == INF: return -1
        total += f[v]
    return total

print(rounds_general([2, 2, 3, 3, 2, 4, 4, 4, 4, 4], [2, 3]), "(= bài gốc)")
print(rounds_general([1] * 7 + [2] * 9, [3, 5]), rounds_general([1] * 7, [4, 6]))
rng = random.Random(152)
for _ in range(300):
    t = [rng.randint(1, 4) for _ in range(rng.randint(1, 40))]
    assert rounds_general(t, [2, 3]) == Solution().minimumRounds(t)
print("với sizes = [2, 3] khớp bài gốc")

4 (= bài gốc)
-1 -1
với sizes = [2, 3] khớp bài gốc


### 15.3 Biến thể C — Mỗi vòng làm 2 hoặc 3 nhiệm vụ **bất kỳ** (không cần cùng mức)

Khi đó mức độ khó không còn quan trọng: chỉ cần `n ≥ 2` và đáp án `⌈n/3⌉`. Một thay đổi nhỏ trong đề làm biến mất cả bước đếm tần suất — đọc đề kỹ!

In [33]:
def rounds_any(tasks):
    n = len(tasks)
    return -1 if n == 1 else (n + 2) // 3
print(rounds_any([2, 3, 3]), "vs bài gốc", Solution().minimumRounds([2, 3, 3]))

1 vs bài gốc -1


### 15.4 Biến thể D — Trả về **lịch cụ thể** (danh sách `(mức, số nhiệm vụ trong vòng)`)

Đã cài ở `build_schedule` (mục 12.3). Ví dụ:

In [34]:
print(build_schedule([2, 2, 3, 3, 2, 4, 4, 4, 4, 4]))
print(build_schedule([2, 3, 3]))

[(2, 3), (3, 2), (4, 3), (4, 2)]
None


### 15.5 Biến thể E — Số vòng **nhiều nhất** (mỗi vòng vẫn 2 hoặc 3 nhiệm vụ cùng mức)

Ngược lại với bài gốc: dùng nhiều vòng-2 nhất. Theo lập luận đổi chỗ ở 6.2 (đổi chiều), lời giải nhiều vòng nhất có tối đa **một** vòng-3: `v` chẵn ⇒ `v/2`; `v` lẻ ⇒ `(v − 3)/2 + 1 = (v − 1)/2`. Tức là `v // 2` trong cả hai trường hợp.

In [35]:
def max_rounds(tasks):
    ans = 0
    for v in Counter(tasks).values():
        if v == 1: return -1
        ans += v // 2
    return ans

def max_rounds_brute(tasks):
    ans = 0
    for v in Counter(tasks).values():
        best = -1
        for b in range(v // 3 + 1):
            if (v - 3 * b) % 2 == 0: best = max(best, b + (v - 3 * b) // 2)
        if best == -1: return -1
        ans += best
    return ans

rng = random.Random(155)
for _ in range(500):
    t = [rng.randint(1, 4) for _ in range(rng.randint(1, 40))]
    assert max_rounds(t) == max_rounds_brute(t)
print(max_rounds([2, 2, 3, 3, 2, 4, 4, 4, 4, 4]), "vòng nhiều nhất cho ví dụ 1 (ít nhất là 4)")

4 vòng nhiều nhất cho ví dụ 1 (ít nhất là 4)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử số vòng-3 | duyệt b = 0..v//3 | O(n) | O(D) | khi chưa thấy công thức |
| 2. DP đổi tiền | f(v) = 1 + min(f(v−2), f(v−3)) | O(n + V) | O(V) | mệnh giá tổng quát |
| 3. BFS | đường ngắn nhất v → 0 | O(n) | O(V) | minh họa |
| 4. Công thức ⌈v/3⌉ | cận dưới + đạt được | O(n) | O(D) | **mặc định** |
| 5/6. Sắp xếp | groupby / hai con trỏ | O(n log n) | O(n) | không dùng băm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumRounds([5,5,5,5,5,5,5,5,5,5,5])` (11 số 5) và `minimumRounds([1,1,2,2,2,3])`?

**Bài 2 (tính tay):** Viết `v = 13` thành tổng các số 2 và 3 với ít số hạng nhất. Có bao nhiêu cách tối ưu (không tính thứ tự)?

**Bài 3 (điền TODO):** Viết `bt3_todo(v)` trả về **số cách** (không tính thứ tự, tức số cặp `(a, b)`) viết `v = 2a + 3b` với `a + b` nhỏ nhất.

**Bài 4 (tự viết lại):** Viết `bt4_todo(tasks)` cài cách 6 (sắp xếp + hai con trỏ) không nhìn code.

**Bài 5 (mini challenge):** Mỗi vòng làm 2 hoặc 3 nhiệm vụ cùng mức, nhưng vòng-3 tốn **chi phí 2**, vòng-2 tốn **chi phí 1**. Viết `bt5_todo(tasks)` trả về **tổng chi phí nhỏ nhất** (hoặc -1).

In [36]:
def bt3_todo(v):
    # TODO: số cặp (a, b) tối ưu
    pass

def bt4_todo(tasks):
    # TODO: sắp xếp + hai con trỏ
    pass

def bt5_todo(tasks):
    # TODO: chi phí vòng-2 = 1, vòng-3 = 2
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: 11 = 3·3 + 2. Mức 2: ⌈11/3⌉ = 4. Mức 3: `4`; mảng hai có mức 3 xuất hiện 1 lần ⇒ `-1`.

**Bài 2.** Mức 1: ⌈13/3⌉ = 5. Mức 2: `a + b = 5` và `2a + 3b = 13` ⇒ `b = 3, a = 2`. Mức 3: chỉ một cách: `3+3+3+2+2`.

**Bài 3.** Mức 1: với `k = a + b` cố định, `2a + 3b = v` ⇒ `b = v − 2k`, `a = 3k − v`. Mức 2: `k = ⌈v/3⌉` (tối ưu); cần `a, b ≥ 0`. Mức 3: chỉ có đúng một cặp ⇒ đáp án 1 (hoặc 0 nếu `v = 1`) — hãy kiểm chứng bằng vét cạn.

**Bài 4.** Mức 1: `a = sorted(tasks)`. Mức 2: `j` chạy hết khối. Mức 3: `v = j − i`, kiểm tra `v == 1`.

**Bài 5.** Mức 1: chi phí mỗi nhiệm vụ: vòng-2 là 1/2, vòng-3 là 2/3 ⇒ vòng-2 **rẻ hơn** tính theo nhiệm vụ. Mức 2: muốn dùng nhiều vòng-2 nhất. Mức 3: `v` chẵn ⇒ `v/2`; `v` lẻ (≥ 3) ⇒ một vòng-3 + `(v−3)/2` vòng-2 ⇒ `2 + (v−3)/2`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `4`, `-1`. **Bài 2:** `3+3+3+2+2` (5 vòng), duy nhất một cách.

In [37]:
assert Solution().minimumRounds([5] * 11) == 4
assert Solution().minimumRounds([1, 1, 2, 2, 2, 3]) == -1

def bt3(v):
    if v == 1: return 0
    k = (v + 2) // 3
    return sum(1 for b in range(k + 1) if 2 * (k - b) + 3 * b == v)

def bt3_brute(v):
    best = None; ways = 0
    for b in range(v // 3 + 1):
        rest = v - 3 * b
        if rest % 2: continue
        r = b + rest // 2
        if best is None or r < best: best, ways = r, 1
        elif r == best: ways += 1
    return ways if best is not None else 0

def bt4(tasks):
    a = sorted(tasks); i = 0; n = len(a); ans = 0
    while i < n:
        j = i
        while j < n and a[j] == a[i]: j += 1
        if j - i == 1: return -1
        ans += (j - i + 2) // 3; i = j
    return ans

def bt5(tasks):
    ans = 0
    for v in Counter(tasks).values():
        if v == 1: return -1
        ans += v // 2 if v % 2 == 0 else 2 + (v - 3) // 2
    return ans

def bt5_brute(tasks):
    ans = 0
    for v in Counter(tasks).values():
        best = None
        for b in range(v // 3 + 1):
            rest = v - 3 * b
            if rest % 2 == 0:
                c = 2 * b + rest // 2
                best = c if best is None else min(best, c)
        if best is None: return -1
        ans += best
    return ans

assert bt3(13) == 1 and all(bt3(v) == bt3_brute(v) for v in range(1, 500))
rng = random.Random(1944)
for _ in range(1500):
    t = [rng.randint(1, 5) for _ in range(rng.randint(1, 40))]
    assert bt4(t) == brute(t)
    assert bt5(t) == bt5_brute(t)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 3: tính duy nhất của lời giải tối ưu đến từ hệ hai phương trình `a + b = k`, `2a + 3b = v` — có đúng một nghiệm `(a, b) = (3k − v, v − 2k)`. Với `k = ⌈v/3⌉` và `v ≥ 2`, cả hai đều không âm.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = 0
for v in Counter(tasks).values():
    if v == 1:
        return -1
    ans += (v + 2) // 3      # ceil(v / 3)
return ans
```

**Checklist:**
- [ ] Đếm tần suất theo **từng** mức (không dùng mảng theo giá trị — `10⁹`).
- [ ] `v == 1` ⇒ `-1` (kiểm tra cho từng mức).
- [ ] Làm tròn **lên**, bằng số học nguyên.
- [ ] Không dùng tham lam "lấy 3 tới cùng" (sai với v ≡ 1 mod 3).
- [ ] Đã thử: `v = 4`, `v = 7`, mảng một phần tử.